# Model Selection

**Topic:** Model Evaluation

In [1]:
import numpy as np
import pandas as pd
import plotly.graph_objects as go
import ipywidgets as widgets
from ipywidgets import IntSlider, FloatSlider, Dropdown, Button, Output, HBox, VBox
from IPython.display import display, HTML, clear_output
from sklearn.datasets import fetch_california_housing
from sklearn.linear_model import LinearRegression, Ridge, LogisticRegression
from sklearn.ensemble import (RandomForestRegressor, RandomForestClassifier,
    GradientBoostingRegressor, GradientBoostingClassifier)
from sklearn.model_selection import (train_test_split, cross_val_score,
    GridSearchCV, RandomizedSearchCV)
from sklearn.metrics import (mean_absolute_error, mean_squared_error, r2_score,
    confusion_matrix, roc_curve, auc, f1_score, precision_score, recall_score)
np.random.seed(42)
from tkh_utils import (PALETTE, FONT, base_layout, plot_confusion_matrix,
    plot_roc_curve, plot_feature_importance, plot_learning_curve)

housing = fetch_california_housing(as_frame=True)
X_reg, y_reg = housing.data, housing.target
X_reg_train, X_reg_test, y_reg_train, y_reg_test = train_test_split(
    X_reg, y_reg, test_size=0.2, random_state=42)

try:
    from sklearn.datasets import fetch_openml
    _hd = fetch_openml(name='heart-disease', version=1, as_frame=True)
    X_cls = _hd.data
    y_cls = (_hd.target.astype(int) > 0).astype(int)
except Exception:
    from sklearn.datasets import load_breast_cancer as _lbc
    _bc = _lbc(as_frame=True)
    X_cls, y_cls = _bc.data, _bc.target
X_cls_train, X_cls_test, y_cls_train, y_cls_test = train_test_split(
    X_cls, y_cls, test_size=0.2, random_state=42)

---
## What you'll explore

By the end of this session you will be able to:

- **Compare** multiple algorithm families on the same dataset using cross-validated metrics
- **Explain** why training time, interpretability, and performance must all factor into model selection
- **Describe** a principled selection workflow that avoids leaking test-set information into the choice

> **Tip:** The fastest model that meets your performance threshold beats the slowest model that exceeds it. Always include training time in your model comparison — in production, the model you can update quickly often beats the one that is 1% more accurate.

---
## How we got here

In `supervised/17_model_comparison.ipynb` you compared supervised algorithms side-by-side on specific tasks. In `ml_concepts/13_interpretability_vs_complexity.ipynb` you placed each algorithm on the interpretability-complexity spectrum.

This notebook brings those threads together: given a new problem, which algorithm family should you start with, and how do you make a principled, evidence-based choice?

---
## Why this matters for data science

Choosing the wrong algorithm wastes training time and may introduce unnecessary complexity. A linear model that achieves R²=0.82 is almost always preferable to a gradient boosting ensemble that achieves R²=0.84 — the interpretability, retraining speed, and debugging ease of the linear model outweigh the 2-point performance gain in most production contexts.

Model selection is also where overfitting to the selection process itself becomes a risk. If you evaluate 20 algorithms on your test set and pick the one that scores highest, your reported performance will be optimistic. All comparisons must use cross validation on training data only.

---
## Try it yourself

In [ ]:
import time
from plotly.subplots import make_subplots
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

out1 = Output()
caption1 = widgets.HTML()

problem_toggle = widgets.ToggleButtons(
    options=[("Regression — California Housing", "reg"),
             ("Classification — Heart Disease", "cls")],
    value="reg",
    description="Problem:",
    style={"description_width": "80px"},
)

REGRESSION_MODELS = {
    "Linear Regression": LinearRegression(),
    "Ridge":             Ridge(alpha=1.0),
    "Random Forest":     RandomForestRegressor(n_estimators=50, random_state=42),
    "Gradient Boosting": GradientBoostingRegressor(n_estimators=50, random_state=42),
}
CLASSIFICATION_MODELS = {
    "Logistic Regression": make_pipeline(StandardScaler(),
        LogisticRegression(max_iter=1000, random_state=42)),
    "Random Forest":       RandomForestClassifier(n_estimators=50, random_state=42),
    "Gradient Boosting":   GradientBoostingClassifier(n_estimators=50, random_state=42),
}
REGRESSION_METRICS = [
    ("R² (higher is better)", "r2"),
    ("MAE (lower is better)", "neg_mean_absolute_error"),
    ("RMSE (lower is better)", "neg_root_mean_squared_error"),
]
CLASSIFICATION_METRICS = [
    ("F1 (higher is better)", "f1"),
    ("ROC-AUC (higher is better)", "roc_auc"),
    ("Precision (higher is better)", "precision"),
    ("Recall (higher is better)", "recall"),
]
METRIC_LABELS = {v: lbl for lbl, v in REGRESSION_METRICS + CLASSIFICATION_METRICS}

algo_select = widgets.SelectMultiple(
    options=list(REGRESSION_MODELS.keys()),
    value=tuple(list(REGRESSION_MODELS.keys())[:3]),
    description="Algorithms:",
    style={"description_width": "80px"},
    layout=widgets.Layout(width="360px", height="90px"),
)
metric_dropdown = Dropdown(
    options=REGRESSION_METRICS,
    value="r2",
    description="Metric:",
    style={"description_width": "80px"},
    layout=widgets.Layout(width="360px"),
)

def render1(change=None):
    is_reg = problem_toggle.value == "reg"
    model_dict = REGRESSION_MODELS if is_reg else CLASSIFICATION_MODELS
    X_data, y_data = (X_reg, y_reg) if is_reg else (X_cls, y_cls)
    metric = metric_dropdown.value
    lower_is_better = metric.startswith("neg_")
    selected = [name for name in algo_select.value if name in model_dict] or list(model_dict.keys())[:1]

    names, means, stds, elapsed_times = [], [], [], []
    for name in selected:
        model = model_dict[name]
        t0 = time.time()
        scores = cross_val_score(model, X_data, y_data, cv=5, scoring=metric)
        elapsed = time.time() - t0
        if lower_is_better:
            scores = -scores
        names.append(name)
        means.append(scores.mean())
        stds.append(scores.std())
        elapsed_times.append(elapsed)

    palette_cycle = [PALETTE["primary"], PALETTE["secondary"], PALETTE["accent"], PALETTE["muted"]]
    bar_colors = [palette_cycle[i % len(palette_cycle)] for i in range(len(names))]

    fig = make_subplots(
        rows=2, cols=1, shared_xaxes=True, row_heights=[0.65, 0.35],
        vertical_spacing=0.12,
        subplot_titles=("Cross-validated score (mean ± std)", "Training time (seconds)"),
    )
    fig.add_trace(go.Bar(
        x=names, y=means, error_y=dict(type="data", array=stds, visible=True),
        marker_color=bar_colors, text=[f"{m:.3f}" for m in means], textposition="auto",
        showlegend=False,
    ), row=1, col=1)
    fig.add_trace(go.Bar(
        x=names, y=elapsed_times, marker_color=bar_colors,
        text=[f"{t:.2f}s" for t in elapsed_times], textposition="auto",
        showlegend=False,
    ), row=2, col=1)

    fig.update_layout(**{k: v for k, v in base_layout(
        title=f"Model Comparison — {METRIC_LABELS[metric]}",
    ).to_plotly_json().items()})
    fig.update_layout(height=480, showlegend=False)

    with out1:
        clear_output(wait=True)
        display(go.FigureWidget(fig))

    best_i = int(np.argmin(means)) if lower_is_better else int(np.argmax(means))
    fastest_i = int(np.argmin(elapsed_times))
    if names[best_i] == names[fastest_i]:
        caption1.value = (
            f"<b>{names[best_i]}</b> is both the best scorer ({means[best_i]:.3f}) and the "
            f"fastest to train ({elapsed_times[best_i]:.2f}s) among the selected algorithms — "
            f"an easy win with no tradeoff."
        )
    else:
        caption1.value = (
            f"<b>{names[best_i]}</b> scores best ({means[best_i]:.3f}), but "
            f"<b>{names[fastest_i]}</b> trains fastest ({elapsed_times[fastest_i]:.2f}s vs. "
            f"{elapsed_times[best_i]:.2f}s). Switch metrics or algorithms above to see how "
            f"that tradeoff shifts."
        )

def on_problem_change(change):
    if problem_toggle.value == "reg":
        algo_select.options = list(REGRESSION_MODELS.keys())
        algo_select.value = tuple(list(REGRESSION_MODELS.keys())[:3])
        metric_dropdown.options = REGRESSION_METRICS
        metric_dropdown.value = "r2"
    else:
        algo_select.options = list(CLASSIFICATION_MODELS.keys())
        algo_select.value = tuple(list(CLASSIFICATION_MODELS.keys()))
        metric_dropdown.options = CLASSIFICATION_METRICS
        metric_dropdown.value = "f1"
    render1()

problem_toggle.observe(on_problem_change, names="value")
algo_select.observe(render1, names="value")
metric_dropdown.observe(render1, names="value")

display(VBox([problem_toggle, algo_select, metric_dropdown, out1, caption1]))
render1()

In [ ]:
out2 = Output()
caption2 = widgets.HTML()

size_dd = Dropdown(
    options=["< 1k rows", "1k-100k rows", "> 100k rows"], value="1k-100k rows",
    description="Dataset size:", style={"description_width": "160px"},
    layout=widgets.Layout(width="420px"),
)
interp_dd = Dropdown(
    options=["High", "Medium", "Low"], value="Medium",
    description="Interpretability need:", style={"description_width": "160px"},
    layout=widgets.Layout(width="420px"),
)
time_dd = Dropdown(
    options=["Seconds", "Minutes", "Hours"], value="Minutes",
    description="Time budget:", style={"description_width": "160px"},
    layout=widgets.Layout(width="420px"),
)
problem_dd = Dropdown(
    options=["Regression", "Binary classification", "Multiclass"], value="Regression",
    description="Problem type:", style={"description_width": "160px"},
    layout=widgets.Layout(width="420px"),
)

# Scores are 1 (poor fit) to 3 (great fit) per criterion; see ml_concepts/13
# for why more complex/opaque models trade away interpretability for accuracy.
ALGO_KB = [
    {"name": "Linear / Logistic Regression",
     "size": {"< 1k rows": 3, "1k-100k rows": 3, "> 100k rows": 3},
     "interp": {"High": 3, "Medium": 2, "Low": 1},
     "time": {"Seconds": 3, "Minutes": 3, "Hours": 2},
     "problem": {"Regression": 3, "Binary classification": 3, "Multiclass": 2},
     "why": "Fastest to train, fully interpretable, but assumes linear relationships"},
    {"name": "Ridge / Lasso",
     "size": {"< 1k rows": 2, "1k-100k rows": 3, "> 100k rows": 3},
     "interp": {"High": 3, "Medium": 2, "Low": 1},
     "time": {"Seconds": 3, "Minutes": 3, "Hours": 2},
     "problem": {"Regression": 3, "Binary classification": 2, "Multiclass": 1},
     "why": "Regularized regression for many/correlated features, stays interpretable"},
    {"name": "Decision Tree",
     "size": {"< 1k rows": 3, "1k-100k rows": 2, "> 100k rows": 1},
     "interp": {"High": 3, "Medium": 2, "Low": 1},
     "time": {"Seconds": 3, "Minutes": 3, "Hours": 3},
     "problem": {"Regression": 2, "Binary classification": 2, "Multiclass": 3},
     "why": "Fully visualizable but overfits small data and is unstable on large data"},
    {"name": "Random Forest",
     "size": {"< 1k rows": 2, "1k-100k rows": 3, "> 100k rows": 2},
     "interp": {"High": 1, "Medium": 3, "Low": 3},
     "time": {"Seconds": 1, "Minutes": 3, "Hours": 3},
     "problem": {"Regression": 3, "Binary classification": 3, "Multiclass": 3},
     "why": "Robust general-purpose baseline; feature importances restore some interpretability"},
    {"name": "Gradient Boosting",
     "size": {"< 1k rows": 1, "1k-100k rows": 3, "> 100k rows": 2},
     "interp": {"High": 1, "Medium": 1, "Low": 3},
     "time": {"Seconds": 1, "Minutes": 2, "Hours": 3},
     "problem": {"Regression": 3, "Binary classification": 3, "Multiclass": 3},
     "why": "Usually the most accurate tabular model, at the cost of time and interpretability"},
    {"name": "SVM",
     "size": {"< 1k rows": 3, "1k-100k rows": 2, "> 100k rows": 1},
     "interp": {"High": 1, "Medium": 2, "Low": 3},
     "time": {"Seconds": 2, "Minutes": 2, "Hours": 2},
     "problem": {"Regression": 2, "Binary classification": 3, "Multiclass": 2},
     "why": "Strong on small, high-dimensional data but slow and opaque on large datasets"},
    {"name": "K-Nearest Neighbors",
     "size": {"< 1k rows": 3, "1k-100k rows": 1, "> 100k rows": 1},
     "interp": {"High": 2, "Medium": 3, "Low": 3},
     "time": {"Seconds": 3, "Minutes": 1, "Hours": 1},
     "problem": {"Regression": 2, "Binary classification": 2, "Multiclass": 2},
     "why": "No training time at all, but prediction slows badly as data size grows"},
    {"name": "Naive Bayes",
     "size": {"< 1k rows": 3, "1k-100k rows": 3, "> 100k rows": 3},
     "interp": {"High": 3, "Medium": 2, "Low": 1},
     "time": {"Seconds": 3, "Minutes": 3, "Hours": 3},
     "problem": {"Regression": 1, "Binary classification": 3, "Multiclass": 3},
     "why": "Extremely fast and interpretable, but only applies to classification"},
]

def render2(change=None):
    size_v, interp_v, time_v, problem_v = (
        size_dd.value, interp_dd.value, time_dd.value, problem_dd.value)
    scored = []
    for algo in ALGO_KB:
        score = (algo["size"][size_v] + algo["interp"][interp_v]
                  + algo["time"][time_v] + algo["problem"][problem_v])
        scored.append((algo["name"], score, algo["why"]))
    scored.sort(key=lambda t: t[1], reverse=True)
    top3 = scored[:3]

    names_r = [t[0] for t in top3][::-1]
    scores_r = [t[1] for t in top3][::-1]
    whys_r = [t[2] for t in top3][::-1]
    colors_r = [PALETTE["primary"], PALETTE["secondary"], PALETTE["accent"]][::-1]

    fig = go.Figure(go.Bar(
        x=scores_r, y=names_r, orientation="h",
        marker_color=colors_r, text=whys_r, textposition="outside",
    ))
    fig.update_layout(**{k: v for k, v in base_layout(
        title="Top 3 recommended algorithms for your answers",
        xaxis_title="Match score (out of 12)",
    ).to_plotly_json().items()})
    fig.update_layout(height=340, xaxis=dict(range=[0, 18]),
                       margin=dict(l=200, r=40, t=60, b=40))

    with out2:
        clear_output(wait=True)
        display(go.FigureWidget(fig))

    winner_name, winner_score, winner_why = top3[0]
    caption2.value = (
        f"<b>Top pick: {winner_name}</b> (score {winner_score}/12) — {winner_why.lower()}. "
        f"Change any of the 4 answers above to see the ranking reshuffle: a stricter "
        f"interpretability requirement or a smaller time budget pushes ensemble methods "
        f"down the list and linear models up."
    )

for ctrl in (size_dd, interp_dd, time_dd, problem_dd):
    ctrl.observe(render2, names="value")

display(VBox([size_dd, interp_dd, time_dd, problem_dd, out2, caption2]))
render2()

---
## What's happening?

Model selection is a search problem in algorithm space. The challenge is doing it honestly:

1. Define your evaluation metric *before* comparing models
2. Use k-fold cross validation on training data only
3. After selection, evaluate the chosen model *once* on the held-out test set
4. Report that final number — never the cross-validation score used for selection

Violating step 2 or 3 inflates your reported performance because you are effectively optimizing for the test set.

| Algorithm | Best for | Avoid when | Typical training time | Interpretability |
|---|---|---|---|---|
| Linear / Logistic Regression | Large datasets, interpretability required | Non-linear relationships | Seconds | Very high |
| Ridge / Lasso | Regression with many features | Strong non-linearities | Seconds | High |
| Random Forest | General purpose, robust to outliers | Very large datasets, memory constrained | Minutes | Medium |
| Gradient Boosting | Best accuracy on tabular data | Small datasets (overfits) | Minutes-hours | Low |
| SVM | High-dimensional data, small-medium datasets | Very large datasets | Minutes | Medium-low |

---
## Real-world example: Comparing four algorithms on California Housing

Four algorithms — Linear Regression, Ridge, Random Forest, and Gradient Boosting — are each evaluated with 5-fold cross validation on California Housing. The chart shows mean R² with ± 1 standard deviation error bars, plus approximate training time.

In [4]:
import numpy as np
import time
import plotly.graph_objects as go
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.model_selection import cross_val_score
from tkh_utils import PALETTE, FONT, base_layout

np.random.seed(42)

models = {
    "Linear Regression": LinearRegression(),
    "Ridge":             Ridge(alpha=1.0),
    "Random Forest":     RandomForestRegressor(n_estimators=50, random_state=42),
    "Gradient Boosting": GradientBoostingRegressor(n_estimators=50, random_state=42),
}

names, means, stds, times = [], [], [], []
for name, model in models.items():
    t0 = time.time()
    scores = cross_val_score(model, X_reg, y_reg, cv=5, scoring='r2')
    elapsed = time.time() - t0
    names.append(name)
    means.append(scores.mean())
    stds.append(scores.std())
    times.append(elapsed)

colors = [PALETTE["primary"], PALETTE["secondary"], PALETTE["accent"], PALETTE["muted"]]

layout = base_layout(
    title="5-Fold Cross-Validated R² — California Housing",
    xaxis_title="Algorithm",
    yaxis_title="R² Score (mean ± std)",
)
layout.update(yaxis=dict(range=[0, 1.1]))
fig = go.Figure(layout=layout)
fig.add_trace(go.Bar(
    x=names, y=means,
    error_y=dict(type='data', array=stds, visible=True),
    marker_color=colors,
    text=[f"{m:.3f}" for m in means],
    textposition="auto",
))
fig.show()

---
## Key takeaway

> **The best model is the simplest one that meets your performance threshold — always include training time and interpretability in the comparison, not just the accuracy metric.**

---
*Next up: `07_learning_curves.ipynb` — diagnosing whether your model needs more data, more complexity, or more regularization*